# Linear Regression & GLMs — AI Lab Instructor Notebook

*Classical ML · Medium*

Build strong, interview-ready intuition for linear regression, regularization, and GLM thinking (loss functions, link functions, assumptions).

**Outcome.** Students can implement linear regression from scratch (closed-form + GD), reason about bias/variance + regularization, and connect GLMs to real ML problems.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Linear Regression & GLMs — Student Lab

Complete all TODOs. Avoid sklearn for core parts.

## Section 0 — Synthetic Dataset (with collinearity)
We generate data where features can be highly correlated to motivate ridge.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_regression(n=250, d=5, noise=0.5, collinear=True):
    X = rng.standard_normal((n, d))
    if collinear and d >= 2:
        X[:, 1] = X[:, 0] + 0.05 * rng.standard_normal(n)
    w_true = rng.standard_normal(d)
    y = X @ w_true + noise * rng.standard_normal(n)
    return X, y, w_true

X, y, w_true = make_regression()
n, d = X.shape
check('shapes', y.shape == (n,))
print('corr(x0,x1)=', np.corrcoef(X[:,0], X[:,1])[0,1])

**Why this works.** # Linear Regression & GLMs — Instructor (Solutions + Rationale)
Runs top-to-bottom.

## Task 2: Closed-form w_hat using solve
*Section: OLS Closed Form*

## Section 1 — OLS Closed Form

### Task 1.1: Closed-form w_hat using solve


**Checkpoint:** Why is explicit inverse discouraged?

**Hints**
- `XtX = X.T@X`, `Xty = X.T@y`, `np.linalg.solve(XtX, Xty)`

In [ ]:
XtX = X.T @ X
Xty = X.T @ y
w_hat = np.linalg.solve(XtX, Xty)

check('w_shape', w_hat.shape == (d,))

In [ ]:
# Checks
check('w_shape', w_hat.shape == (d,))

## Task 3: Evaluate fit + residuals
*Section: OLS Closed Form*

### Task 1.2: Evaluate fit + residuals
Compute:
- predictions y_pred
- MSE
- residual mean and std

**Interview Angle:** What does a structured residual pattern imply (e.g., nonlinearity)?

In [ ]:
y_pred = X @ w_hat
mse = np.mean((y_pred - y) ** 2)
resid = y - y_pred
print('mse', mse, 'resid_mean', resid.mean(), 'resid_std', resid.std())
check('finite', np.isfinite(mse))

In [ ]:
# Checks
check('finite', np.isfinite(mse))

**Why this works.** scaling affects Lipschitz constant of gradient -> convergence speed depends on lr and conditioning.

## Task 4: Implement MSE loss + gradient
*Section: Gradient Descent*

## Section 2 — Gradient Descent

### Task 2.1: Implement MSE loss + gradient

Loss = mean((Xw-y)^2), grad = (2/n) X^T(Xw-y)


**FAANG gotcha:** shapes and constants.

In [ ]:
def mse_loss_and_grad(X, y, w):
    r = X @ w - y
    loss = float(np.mean(r * r))
    grad = (2.0 / X.shape[0]) * (X.T @ r)
    return loss, grad

w0 = np.zeros(d)
loss0, g0 = mse_loss_and_grad(X, y, w0)
check('grad_shape', g0.shape == (d,))
check('finite_loss', np.isfinite(loss0))

In [ ]:
# Checks
check('grad_shape', g0.shape == (d,))
check('finite_loss', np.isfinite(loss0))

**Why this works.** ridge reduces variance by shrinking weights; helps under collinearity/ill-conditioning.

## Task 5: Train with GD + compare to closed-form
*Section: Gradient Descent*

### Task 2.2: Train with GD + compare to closed-form


**Checkpoint:** How does feature scaling affect GD?

In [ ]:
def train_gd(X, y, lr=0.05, steps=500):
    w = np.zeros(X.shape[1])
    losses = []
    for _ in range(steps):
        loss, g = mse_loss_and_grad(X, y, w)
        losses.append(loss)
        w = w - lr * g
    return w, losses

w_gd, losses = train_gd(X, y, lr=0.05, steps=500)
print('final_loss', losses[-1])
print('||w_gd-w_hat||', np.linalg.norm(w_gd - w_hat))
check('loss_decreases', losses[-1] <= losses[0])

In [ ]:
# Checks
check('loss_decreases', losses[-1] <= losses[0])

**Why this works.** ridge reduces variance by shrinking weights; helps under collinearity/ill-conditioning.

## Task 6: Ridge closed-form + Ridge GD
*Section: Ridge Regression (L2)*

## Section 3 — Ridge Regression (L2)

### Task 3.1: Ridge closed-form + Ridge GD
Use a matching objective for both methods:

L(w) = 0.5 ||Xw - y||^2 + (λ/2) ||w||^2

- Closed-form: `w = (X^T X + λI)^{-1} X^T y`
- Gradient: `∇L = X^T(Xw - y) + λw`

Implement both and compare the learned weights.

**Interview Angle:** Why is objective consistency important when comparing solvers?

In [ ]:
def ridge_solve(X, y, lam):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)

def ridge_loss_and_grad(X, y, w, lam):
    r = X @ w - y
    loss = 0.5 * float(r @ r) + 0.5 * lam * float(w @ w)
    grad = X.T @ r + lam * w
    return loss, grad

def train_ridge_gd(X, y, lam=1.0, lr=0.002, steps=1500):
    w = np.zeros(X.shape[1])
    losses = []
    for _ in range(steps):
        loss, g = ridge_loss_and_grad(X, y, w, lam)
        losses.append(loss)
        w = w - lr * g
    return w, losses

lam = 1.0
w_ridge_closed = ridge_solve(X, y, lam=lam)
w_ridge_gd, ridge_losses = train_ridge_gd(X, y, lam=lam, lr=0.002, steps=1500)

print('ridge_final_loss', ridge_losses[-1])
print('||w_gd - w_closed||', np.linalg.norm(w_ridge_gd - w_ridge_closed))
check('ridge_shape', w_ridge_closed.shape == (d,))
check('ridge_loss_decreases', ridge_losses[-1] <= ridge_losses[0])
check('ridge_close_to_closed', np.linalg.norm(w_ridge_gd - w_ridge_closed) < 0.1)

In [ ]:
# Checks
check('ridge_shape', w_ridge_closed.shape == (d,))
check('ridge_loss_decreases', ridge_losses[-1] <= ridge_losses[0])
check('ridge_close_to_closed', np.linalg.norm(w_ridge_gd - w_ridge_closed) < 0.1)

**Why this works.** ridge reduces variance by shrinking weights; helps under collinearity/ill-conditioning.

## Task 7: Bias/variance demo with train/test split
*Section: Ridge Regression (L2)*

### Task 3.2: Bias/variance demo with train/test split

**Checkpoint:** why can test error improve even when train error worsens?

In [ ]:
# TODO
idx = rng.permutation(n)
train = idx[: int(0.5*n)]
test = idx[int(0.5*n):]
Xtr, ytr = X[train], y[train]
Xte, yte = X[test], y[test]

lams = [0.0, 0.1, 1.0, 10.0]
results = []
for lam in lams:
    w = ridge_solve(Xtr, ytr, lam=lam) if lam > 0 else np.linalg.solve(Xtr.T@Xtr, Xtr.T@ytr)
    tr_mse = np.mean((Xtr@w - ytr)**2)
    te_mse = np.mean((Xte@w - yte)**2)
    results.append((lam, tr_mse, te_mse))
print('lam, train_mse, test_mse')
for r in results:
    print(r)

**Why this works.** ridge reduces variance by shrinking weights; helps under collinearity/ill-conditioning.

## Task 8: GLM intuition table
*Section: GLM Intuition*

## Section 4 — GLM Intuition

### Task 4.1: Match tasks to (distribution, link)
Fill in a table for:
- regression
- binary classification
- count prediction

**Explain:** what changes when you go from OLS to a GLM?

| Problem | Target type | Distribution | Link | Loss |
|---|---|---|---|---|
| House price | continuous | ? | ? | ? |
| Fraud | binary | ? | ? | ? |
| Clicks per user | count | ? | ? | ? |

**Hints**
- Gaussian + identity link -> linear regression (MSE)
- Bernoulli + logit link -> logistic regression (log loss)
- Poisson + log link -> count model

**Explain:** What changes when you go from OLS to a GLM?

In [ ]:
# Conceptual section: no coding required.

**Why this works.** GLMs keep a linear predictor but model different target distributions via an appropriate link function.

| Problem | Target type | Distribution | Link | Loss |
|---|---|---|---|---|
| House price | continuous | Gaussian (Normal) | Identity | MSE |
| Fraud | binary | Bernoulli | Logit (sigmoid inverse) | Binary cross-entropy |
| Clicks per user | count | Poisson | Log | Poisson negative log-likelihood |

Going from OLS to a GLM changes two things:
1. You choose a target distribution that matches the data type.
2. You use a link function to map the linear predictor into the valid range of the mean.